# 05. Key Insights and Business Recommendations

Цей ноутбук узагальнює основні висновки за результатами EDA, SQL-аналізу, статистичної перевірки та baseline ML-моделі.

Мета — перетворити аналітичні результати на чіткі бізнес-інсайти для ризик-сегментації та скорингової логіки.

## Baseline reference

> Загальний default rate по вибірці: 8.07%
> Усі порівняння сегментів нижче використовують це значення як орієнтир.

---

## 1. Key Insights

### 1.1 Цільова змінна є незбалансованою

Лише 8.07% клієнтів у вибірці мають дефолт (TARGET = 1), тоді як понад 91.93% — ні.

Цей дисбаланс безпосередньо впливає на оцінку моделі: модель, яка прогнозує "No default" для кожного клієнта, все одно матиме ~92% accuracy — тому accuracy сама по собі є хибним показником.

Висновок: ефективність моделі потрібно оцінювати через Recall, Precision, F1-score та ROC-AUC, а не через accuracy.

---

### 1.2 Вік є одним із найсильніших демографічних сигналів ризику

| Вікова група | Default rate | vs. baseline |
|---|---|---|
| 20–29 | 11.47% | +3.40 pp |
| 30–39 | 9.20% | +1.13 pp |
| 40–49 | 7.85% | −0.22 pp |
| 50–59 | 6.40% | −1.67 pp |
| 60+ | 5.10% | −2.97 pp |

Наймолодший сегмент (20–29) має default rate у 1.4 рази вище за загальний показник по вибірці.

Можливе пояснення: коротша кредитна історія, менш стабільний дохід, менші фінансові резерви.

---

### 1.3 Співвідношення кредиту до доходу є важливим індикатором ризику, але не лінійним

| Credit income group | Default rate | vs. baseline |
|---|---|---|
| < 1 | 6.20% | −1.87 pp |
| 1–2 | 7.50% | −0.57 pp |
| 2–3 | 9.80% | +1.73 pp |
| 3–5 | 10.10% | +2.03 pp |
| > 5 | 8.30% | +0.23 pp |

Найвищий default rate спостерігається у групах 2–3 та 3–5, де сума кредиту у 2–5 разів перевищує річний дохід клієнта.

Зниження показника у групі > 5 може відображати інший профіль клієнта (наприклад, клієнти з високим доходом та великими кредитами) — цю групу варто аналізувати окремо, а не вважати менш ризиковою.

---

### 1.4 Попередні відхилені заявки є статистично значущим сигналом ризику

Клієнти, які мали хоча б одну попередню заявку зі статусом Refused, мають суттєво вищий default rate:

| Група | Default rate |
|---|---|
| Без попередніх Refused | 6.98% |
| З попередніми Refused | 10.32% |
| Різниця | +3.34 pp |

Результат статистичної перевірки: тест Манна-Уітні підтвердив, що різниця є статистично значущою (p < 0.05).

Це означає, що патерн не є випадковим і може надійно використовуватися як ознака ризику у скоринговій моделі.

---

### 1.5 Статус попередньої заявки прогнозує поточний ризик дефолту

| Статус попередньої заявки | Default rate |
|---|---|
| Refused | 12.00% |
| Canceled | 9.40% |
| Unused offer | 8.80% |
| Approved | 7.59% |

Клієнти, чиї попередні заявки були Refused, мають default rate у 1.58 рази вище, ніж клієнти з Approved-історією.

---

### 1.6 Тип попереднього кредитного продукту є додатковим сигналом ризику

| Тип попереднього продукту | Default rate |
|---|---|
| Revolving loans | 10.47% |
| Consumer loans | 8.20% |
| Cash loans | 7.90% |

Revolving loans у попередніх заявках пов'язані з вищим поточним default rate. Цей сегмент потребує окремого аналізу в скоринговій логіці.

---

### 1.7 Offer gap є слабким, але потенційно корисним сигналом

Клієнти, яким у попередніх заявках було погоджено **суму вищу за запитану**, мають дещо вищий default rate:

| Offer gap group | Default rate |
|---|---|
| Approved < Requested | 7.80% |
| Approved ≈ Requested | 8.00% |
| Approved > Requested | 8.10% |

Різниця є невеликою та статистично не підтверджена як самостійний сигнал. Однак ця ознака може додавати цінність у складі комбінованого набору ознак у майбутній ML-моделі.

---

### 1.8 Baseline ML-модель виявила основну проблему моделювання

Як стартова точка було навчено просту Logistic Regression.

| Метрика | Результат |
|---|---|
| Accuracy | ~92% |
| Recall (клас дефолту) | Дуже низький |
| ROC-AUC | Помірний |

Висновок: висока accuracy є хибним показником — модель навчилася прогнозувати "No default" майже для всіх клієнтів, оскільки це більшість у вибірці.

Це підтверджує, що дисбаланс класів необхідно явно враховувати, перш ніж будь-яка модель зможе надійно визначати ризикових клієнтів.

---

## 2. Combined Risk View

Таблиця нижче узагальнює сегменти з найвищим ризиком дефолту за ключовими факторами аналізу.

| Фактор | Ризиковий сегмент | Default rate | vs. baseline |
|---|---|---|---|
| Вік | 20–29 років | 11.47% | +3.40 pp |
| Credit income ratio | 3–5 | 10.10% | +2.03 pp |
| Попередній статус | Refused | 12.00% | +3.93 pp |
| Попередній продукт | Revolving loans | 10.47% | +2.40 pp |
| Наявність Refused | Є хоча б одна | 10.32% | +3.34 pp |

Ключовий висновок: жоден фактор сам по собі не є вирішальним.
Найвищий ризик концентрується у клієнтів, які одночасно потрапляють до кількох ризикових сегментів.

---

## 3. Business Recommendations

### 3.1 Враховувати історію попередніх заявок у скоринговій логіці

Наявність попередніх Refused-заявок є статистично підтвердженим сигналом ризику (+3.34 pp до baseline).

Рекомендація: додати ознаку has_previous_refused як обов'язкову до скорингового пайплайну або ризик-сегментації.

---

### 3.2 Окремо аналізувати клієнтів із попередніми Revolving loans

Default rate для цього сегменту — 10.47%, що на 2.40 pp вище за baseline.

Рекомендація: не відмовляти автоматично, але проводити більш детальну перевірку для клієнтів із попередніми revolving-продуктами.

---

### 3.3 Використовувати співвідношення, а не лише абсолютні суми

Сума кредиту або дохід окремо не дають повної картини фінансового навантаження клієнта.

Рекомендація: використовувати credit_income_ratio та annuity_income_ratio як основні індикатори навантаження.
Особливу увагу приділяти групам 2–3 та 3–5.

---

### 3.4 Приділяти додаткову увагу молодим клієнтам

Вікова група 20–29 має default rate 11.47% — найвищий серед усіх вікових сегментів.

Рекомендація: для молодих клієнтів без кредитної історії розглядати нижчі початкові кредитні ліміти або додаткові критерії верифікації.

---

### 3.5 Не приймати рішення на основі одного фактора

Ризик дефолту формується комбінацією демографічних, фінансових та історичних факторів.

Рекомендація: використовувати комплексний підхід — поєднувати вік, credit_income_ratio, тип доходу, освіту та історію попередніх заявок при оцінці ризику клієнта.

---

### 3.6 Обережно інтерпретувати малі сегменти

Деякі групи показують дуже високий або низький default rate, але при малій кількості клієнтів цей результат може бути нестабільним.

Рекомендація: не використовувати висновки по малих сегментах як основний орієнтир без додаткової статистичної перевірки.

---

### 3.7 Покращити ML-модель у наступних ітераціях

Baseline Logistic Regression підтвердила проблему дисбалансу класів, але не є готовим рішенням для прогнозування дефолту.

Рекомендація: у наступних ітераціях:
- Додати більше ознак із previous_application та інших таблиць
- Застосувати методи роботи з дисбалансом: class_weight, SMOTE
- Змінити поріг класифікації для підвищення Recall
- Порівняти кілька моделей: Random Forest, XGBoost, LightGBM
- Оцінювати результати через ROC-AUC та F1-score, а не accuracy

---

### 3.8 Розширити аналіз додатковими таблицями датасету

Поточна версія використовує лише дві таблиці: application_train та previous_application.

Рекомендація: для глибшого аналізу додати:

| Таблиця | Що додасть |
|---|---|
| bureau | Зовнішня кредитна історія клієнта |
| bureau_balance | Динаміка зовнішніх кредитів по місяцях |
| installments_payments | Платіжна поведінка по попередніх кредитах |
| credit_card_balance | Використання кредитних карток |
| POS_CASH_balance | Поведінка по POS-кредитах |

## Final note

Основний фокус аналізу — дві таблиці: `application_train` та `previous_application`. Це дозволило проаналізувати базові характеристики клієнтів, фінансове навантаження та історію попередніх заявок.

Розширення аналізу на додаткові таблиці датасету дасть змогу глибше зрозуміти фактори дефолту, покращити набір ознак та побудувати надійнішу ML-модель у наступних ітераціях.